In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
from utils.seed import SeedSetter
from utils.datasets import TimeSeriesDataset
from torch.utils.data import DataLoader

In [4]:
from gdmh.mia import PolynomType


class Config:
    seed: int = 42
    test_size: float = 0.2
    p_average: int = 3
    k_best: int = 5
    polynomial_type: PolynomType = PolynomType.QUADRATIC
    limit: float = 0.01
    max_iters: int = 100

In [5]:
SeedSetter(Config.seed)

Random seed set to 42


In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [7]:
data = pd.read_csv("../data/turbines_train.csv")
data = data.drop(["Unnamed: 0", "Time"], axis=1)

In [8]:
data.head(5)

,Location,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Power
0,1,28.2796,84.664205,24.072595,1.605389,1.267799,145.051683,161.057315,1.336515,0.163496
1,1,28.1796,85.664205,24.272595,2.225389,3.997799,150.051683,157.057315,4.336515,0.142396
2,1,26.5796,90.664205,24.072595,1.465389,2.787799,147.051683,149.057315,3.136515,0.121396
3,1,27.1796,87.664205,23.872595,1.465389,2.697799,57.051683,104.057315,1.536515,0.100296
4,1,27.0796,87.664205,23.672595,2.635389,4.437799,57.051683,83.057315,3.936515,0.079296


In [9]:
data["Location"].unique()

array([1, 2, 3, 4])

In [10]:
scaler = StandardScaler()
features = data.drop(['Power', 'Location'], axis=1)
scaled_features = scaler.fit_transform(features)
scaled_data = pd.DataFrame(scaled_features, columns=features.columns)
scaled_data['Power'] = data['Power'].values
scaled_data['Location'] = data['Location'].values

In [11]:
scaled_data = pd.concat([pd.get_dummies(scaled_data.drop(["Power"], axis=1), columns=['Location']), data["Power"]], axis=1)
scaled_data.head()

,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Location_1,Location_2,Location_3,Location_4,Power
0,-0.804021,0.837857,-0.562194,-1.355386,-1.850563,-0.555885,-0.391588,-1.845008,1,0,0,0,0.163496
1,-0.808581,0.896680,-0.552661,-1.049536,-0.957421,-0.505925,-0.431150,-1.017793,1,0,0,0,0.142396
2,-0.881539,1.190795,-0.562194,-1.424449,-1.353283,-0.535901,-0.510276,-1.348679,1,0,0,0,0.121396
3,-0.854179,1.014326,-0.571727,-1.424449,-1.382727,-1.435186,-0.955355,-1.789861,1,0,0,0,0.100296
4,-0.858739,1.014326,-0.581260,-0.847281,-0.813472,-1.435186,-1.163059,-1.128089,1,0,0,0,0.079296


In [12]:
scaled_data['power_log'] = np.log1p(scaled_data['Power'])
scaled_data = scaled_data.drop(['Power'], axis=1)
scaled_data.head()

,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Location_1,Location_2,Location_3,Location_4,power_log
0,-0.804021,0.837857,-0.562194,-1.355386,-1.850563,-0.555885,-0.391588,-1.845008,1,0,0,0,0.151429
1,-0.808581,0.896680,-0.552661,-1.049536,-0.957421,-0.505925,-0.431150,-1.017793,1,0,0,0,0.133128
2,-0.881539,1.190795,-0.562194,-1.424449,-1.353283,-0.535901,-0.510276,-1.348679,1,0,0,0,0.114574
3,-0.854179,1.014326,-0.571727,-1.424449,-1.382727,-1.435186,-0.955355,-1.789861,1,0,0,0,0.095579
4,-0.858739,1.014326,-0.581260,-0.847281,-0.813472,-1.435186,-1.163059,-1.128089,1,0,0,0,0.076309


In [13]:
print(
    f"features shape: {scaled_data.iloc[:, :-1].to_numpy().shape}, target shape: {scaled_data.iloc[:, -1].to_numpy().shape}")

features shape: (140160, 12), target shape: (140160,)


In [14]:
split_size = int(len(scaled_data) * 0.75)
train_data = scaled_data[:split_size]
test_data = scaled_data[split_size:]

In [15]:
window_size = 10
train_dataset = TimeSeriesDataset(data=train_data, window_size=window_size)
test_dataset = TimeSeriesDataset(data=test_data, window_size=window_size)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [17]:
train_x, train_y = next(iter(train_loader))
test_x, test_y = next(iter(test_loader))

In [18]:
def criterion(y_pred, y_true) -> float:
    return np.mean((y_true - y_pred) ** 2)

In [19]:
from gdmh.mia import Mia

mia = Mia(test_size=Config.test_size, 
          p_average=Config.p_average, 
          k_best=Config.k_best,
          polynomial_type=Config.polynomial_type,
          criterion=criterion,
          seed=Config.seed,
          max_iters=Config.max_iters,
          limit=Config.limit)

In [20]:
mia.fit(train_x.numpy(), train_y.numpy())

Iteration 1 completed: average criterion 0.018157580867409706
Iteration 2 completed: average criterion 2.728811264038086
Stopping: improvement below limit.


In [21]:
test_predictions = mia.predict(test_x.numpy())

In [24]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(test_y.numpy(), test_predictions)
mae = mean_absolute_error(test_y.numpy(), test_predictions)
r2 = r2_score(test_y.numpy(), test_predictions)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error (MSE): 0.0114
Mean Absolute Error (MAE): 0.0864
R2 Score: 0.6109
